# F3-matrices — Session 3: Rank, Independence, and Outer Products

*One class session, roughly 85 minutes. Prerequisite: Sessions 1–2 of
this unit and F2-vectors (dot products, orthogonality).*

**This session:** when one vector in a list is secretly built from the
others (linear independence), the set of everything a list can reach
(span), the single number that measures a matrix's genuine directional
content (rank), what rank decides about undoing a map (invertibility),
and the rank-1 atoms — outer products — from which every matrix can be
assembled, including how *few* atoms suffice.
Plus a fully worked exam-style multiple-choice problem in the real
paper's normal-form register.

Try every checkpoint by hand first, then verify with NumPy.
Answers are collected at the end of this notebook.

In [1]:
import numpy as np

## 1. Building One Vector From Others

**Motivation.**
Three sensors each record a direction; unbeknownst to you, sensor 3's
wire is soldered to the sum of the first two.
Its data adds *nothing*: every reading it will ever produce is
computable from the others.
Linear algebra's name for "adds nothing" is *dependence*, and detecting
it is a skill the exam tests directly.

**Definition (operational).**
A list of vectors $v_1, \dots, v_r$ is **linearly dependent** when at
least one of them is a combination of the others — some $v_p$ equals
$\sum_{q \ne p} c_q v_q$.
The list is **linearly independent** when no listed vector can be built
from the rest: each one carries genuinely new directional information.

**Worked examples.**

- $(2, 1)$ and $(1, 2)$: independent — every multiple $c\,(2,1)$ has
  its entries in ratio $2\!:\!1$, and $(1,2)$ does not.
- $(1, -1, 0)$, $(0, 1, -1)$, $(1, 0, -1)$: dependent — the third is
  the sum of the first two.
- Any list containing the zero vector: dependent — $0$ is the empty
  combination ($0 = 0 \cdot v_1 + \cdots$) of the others.
- A single nonzero vector: independent (nothing else to build it
  from).

**The equivalent zero-combination test.**
The list is dependent exactly when some combination
$c_1 v_1 + \cdots + c_r v_r = 0$ has *not all* $c_q = 0$: if
$c_p \ne 0$, solve for $v_p$ — it is built from the others.
This restatement is the workhorse in proofs; the operational reading is
the workhorse at the whiteboard.

**Small-list shortcuts worth memorizing.**

- Two vectors: dependent ⇔ one is a scalar multiple of the other.
- More vectors than dimensions (three 2-D vectors, four 3-D vectors,
  …): *always* dependent — Section 3 explains why via rank.

In [2]:
v1 = np.array([1., -1., 0.])
v2 = np.array([0., 1., -1.])
v3 = np.array([1., 0., -1.])
print("v1 + v2      :", v1 + v2)
print("v3           :", v3)
print("build gap    :", np.abs((v1 + v2) - v3).max(), " -> v3 adds nothing")

# Two-vector shortcut: multiple or not?
a, b = np.array([3., -6., 9.]), np.array([-1., 2., -3.])
print("a / b entrywise:", a / b, " -> constant ratio -3: dependent")

v1 + v2      : [ 1.  0. -1.]
v3           : [ 1.  0. -1.]
build gap    : 0.0  -> v3 adds nothing
a / b entrywise: [-3. -3. -3.]  -> constant ratio -3: dependent


### Checkpoint 1

1. $(3, 1)$ and $(6, 2)$: independent or dependent?
   One line.
2. $(1, 0, 1)$, $(0, 1, 0)$, $(1, 1, 1)$: show these are dependent by
   exhibiting the build.
3. Why is *any* list containing the zero vector automatically
   dependent?

## 2. The Span: Every Vector You Can Reach

**Motivation.**
Section 1 asks "is anyone redundant?"; the dual question is "what can
the whole list *do*?"
Both questions meet in Section 3.

**Definition.**
The **span** of $v_1, \dots, v_r$ is the set of *all* combinations
$c_1 v_1 + \cdots + c_r v_r$ — everything reachable using the list as
building blocks.

**Geometry (in $\mathbb{R}^3$).**

- One nonzero vector: its span is the **line** through it.
- Two independent vectors: a **plane**.
- Three independent vectors: all of $\mathbb{R}^3$.
- Adding a *dependent* vector to a list never changes the span — it was
  already reachable.

**Worked example (a membership test you can compute).**
What is the span of $u = (1, 2, 1)$ and $w = (0, 1, 3)$?
A reachable vector is $y = a\,u + b\,w = (a,\; 2a + b,\; a + 3b)$.
Eliminate the knobs: $a = y_0$, then $b = y_1 - 2y_0$, and the last
entry must obey
$$y_2 = a + 3b = y_0 + 3(y_1 - 2y_0) = 3y_1 - 5y_0 .$$
So the span is exactly the plane of vectors satisfying
$y_2 = 3y_1 - 5y_0$ — a one-line membership test.
$(1, 3, 4)$: $3 \cdot 3 - 5 \cdot 1 = 4$ ✓ reachable (indeed
$u + w$).
$(0, 0, 1)$: $3 \cdot 0 - 5 \cdot 0 = 0 \ne 1$ ✗ unreachable.
Keep this example — it returns in Section 4 as an invertibility
argument.

In [3]:
u = np.array([1., 2., 1.])
w = np.array([0., 1., 3.])


def in_span_uw(y):
    # membership test derived by eliminating the combination knobs
    return np.isclose(y[2], 3 * y[1] - 5 * y[0], atol=1e-12, rtol=0)


for y in [np.array([1., 3., 4.]), np.array([0., 0., 1.]),
          2.5 * u - 4.0 * w]:
    print(y, "reachable" if in_span_uw(y) else "UNREACHABLE")

[1. 3. 4.] reachable
[0. 0. 1.] UNREACHABLE
[ 2.5  1.  -9.5] reachable


### Checkpoint 2

1. Describe the span of $(1, 0, 0)$ and $(0, 0, 1)$ in $\mathbb{R}^3$.
   Is $(2, 0, -5)$ in it?
   $(0, 1, 0)$?
2. Using the worked test: is $(2, 1, -7)$ in the span of $u = (1,2,1)$
   and $w = (0,1,3)$?
3. You append $u + 2w$ to the list $\{u, w\}$.
   What happens to the span, and why?

## 3. Rank: Counting Independent Directions

**Motivation.**
A matrix may parade many rows, yet carry few genuinely different
directions — like the wired-together sensors.
Rank is the honest count.

**Definition.**
The **rank** of a matrix is the number of independent directions among
its rows — equivalently, the largest independent sub-list you can select
from them; equivalently again, the number of independent directions
needed to span all the rows.

**A usable fact (stated, not proved here):**
counting with the *columns* gives the **same number**.
Row count of independent directions = column count of independent
directions, always.
Use whichever side is easier to eyeball — this equality is an exam-grade
tool, not a curiosity.

**Worked reads (by inspection).**

- $\begin{pmatrix} 2 & 4 \\ 1 & 2 \end{pmatrix}$: row 0 is twice
  row 1 → one independent direction → rank 1.
- $\begin{pmatrix} 1 & 1 & 0 \\ 0 & 1 & 1 \\ 1 & 2 & 1 \end{pmatrix}$:
  row 2 = row 0 + row 1, rows 0 and 1 not multiples of each other →
  rank 2.
- The zero matrix: rank 0 (no directions at all).
- $\begin{pmatrix} 1 & 0 & 2 \\ 0 & 1 & -1 \end{pmatrix}$: two rows,
  not multiples → rank 2 — and the column view agrees: three columns
  living in 2-D can muster at most 2 independent directions
  (Section 1's shortcut), and columns $(1,0)$, $(0,1)$ already deliver
  2.

**Bounds that come free.**
$\operatorname{rank}(M) \le \min(\text{rows}, \text{columns})$ — you
cannot have more independent directions than vectors, nor than the
dimension they live in.
A matrix achieving the bound is called **full rank**.

**One more usable fact (for products):**
$\operatorname{rank}(AB) \le \min(\operatorname{rank} A, \operatorname{rank} B)$ —
a pipeline cannot *create* directional content its stages lack.
(Plausibility: every column of $AB$ is $A$ applied to a column of $B$,
hence lives in the span of $A$'s columns.)

In [4]:
# The independent-direction count, verified by brute reasoning in code:
# for the 3x3 example, check that row 2 is exactly row 0 + row 1.
Mex = np.array([[1., 1., 0.], [0., 1., 1.], [1., 2., 1.]])
print("row2 - (row0 + row1):", Mex[2] - (Mex[0] + Mex[1]))     # zeros -> dependent
print("row0 vs row1 ratio  :", Mex[0] / np.where(Mex[1] == 0, np.nan, Mex[1]))
print("  -> not a constant ratio: rows 0, 1 independent; rank = 2")

row2 - (row0 + row1): [0. 0. 0.]
row0 vs row1 ratio  : [nan  1.  0.]
  -> not a constant ratio: rows 0, 1 independent; rank = 2


### Checkpoint 3

1. Rank of $\begin{pmatrix} 1 & 3 \\ 2 & 6 \end{pmatrix}$?
2. Rank of $\begin{pmatrix} 1 & 0 & 1 \\ 0 & 1 & 1 \\ 1 & 1 & 2 \end{pmatrix}$?
   Name the dependence you used.
3. What is the largest possible rank of a $(3, 5)$ matrix, and which
   side (rows or columns) forces that bound?

## 4. Invertibility Through the Rank Lens

**Motivation.**
An **inverse** of a square machine $A$ is an undo machine: a map that
returns every output to the input it came from.
When does an undo machine exist?
Rank answers completely — and lets you *decide* invertibility without
ever *computing* an inverse.
(Computing inverses is genuinely out of this unit's scope, and the
exam's register bans the library shortcuts anyway; rank reasoning is
both the required tool and the sufficient one.)

**The criterion.**
For a square $n \times n$ matrix $A$:
$$A \text{ is invertible} \iff \operatorname{rank}(A) = n
  \quad (\text{full rank}).$$

**Why rank $< n$ kills the undo — two independent arguments.**
Take the fresh example
$$A = \begin{pmatrix} 1 & 0 & 1 \\ 2 & 1 & 3 \\ 1 & 3 & 4 \end{pmatrix},$$
whose columns are $c_0 = (1,2,1)$, $c_1 = (0,1,3)$,
$c_2 = (1,3,4) = c_0 + c_1$: rank 2.

- **Collisions.**
  The dependence gives a nonzero input the machine crushes to zero:
  $x^\* = (1, 1, -1)$ has
  $Ax^\* = c_0 + c_1 - c_2 = 0$.
  Then *any* two inputs differing by $x^\*$ — say $v$ and
  $v + x^\*$ — produce identical outputs.
  An undo machine receiving that output cannot know which input to
  return: no inverse.
- **Unreachable targets.**
  Every output $Ax = x_0 c_0 + x_1 c_1 + x_2 c_2$ lies in the span of
  the columns — here the Section 2 plane $y_2 = 3y_1 - 5y_0$ (the
  dependent $c_2$ adds nothing).
  The target $(0, 0, 1)$ violates the plane equation: *no input ever
  produces it*.
  An inverse would have to answer "which input gives $(0,0,1)$?" — the
  question has no answer: no inverse.

Either failure alone is fatal; rank $< n$ always triggers **both**.
Conversely, at full rank the columns' only zero-combination is the
trivial one (no collisions: $Au = Av$ would make $A(u - v) = 0$ with
$u - v \ne 0$, a nonzero zero-combination of independent columns —
impossible), and $n$ independent columns span all of $\mathbb{R}^n$
(no unreachable targets) — the undo is well defined both ways.

**Non-square and products.**
Only square matrices are candidates for a two-sided undo.
And Section 3's product fact settles a favorite exam trap instantly: a
$4 \times 4$ matrix assembled as $AB$ with $A\,(4, 2)$, $B\,(2, 4)$ has
rank $\le 2 < 4$ — never invertible, whatever the entries.

In [5]:
A4 = np.array([[1., 0., 1.], [2., 1., 3.], [1., 3., 4.]])

# Collision: the column dependence c2 = c0 + c1 crushes (1, 1, -1) to zero.
x_star = np.array([1., 1., -1.])
print("A x* :", (A4 * x_star).sum(axis=1))              # ~0: crushed

SEED = 20260804
v = np.random.default_rng(SEED).normal(0, 1, 3)
out_v = (A4 * v).sum(axis=1)
out_v2 = (A4 * (v + x_star)).sum(axis=1)
print("collision gap between A v and A(v + x*):", np.abs(out_v - out_v2).max())

# Unreachable target: every output obeys y2 = 3 y1 - 5 y0; (0,0,1) does not.
y = out_v
print("plane residue of a real output:", y[2] - (3 * y[1] - 5 * y[0]))   # ~0
t = np.array([0., 0., 1.])
print("plane residue of target (0,0,1):", t[2] - (3 * t[1] - 5 * t[0]))  # 1 != 0

A x* : [0. 0. 0.]
collision gap between A v and A(v + x*): 4.996003610813204e-16
plane residue of a real output: -4.440892098500626e-16
plane residue of target (0,0,1): 1.0


### Checkpoint 4

1. Decide by rank: is $\begin{pmatrix} 1 & 3 \\ 2 & 6 \end{pmatrix}$
   invertible?
   If not, exhibit a nonzero collision input $x$ with $Ax = 0$.
2. One line: why can a full-rank square machine never send two
   *different* inputs to the same output?
3. $A$ is $5 \times 3$ and $B$ is $3 \times 5$.
   Can $AB$ (a $5 \times 5$ matrix) be invertible?
   Cite the fact you used.

## 5. Outer Products: The Rank-1 Atoms

**Motivation.**
What does the *simplest possible* nonzero matrix look like — the one
with only a single direction's worth of content?
It is manufactured from two vectors, and the exam builds problems from
these atoms every year.

**Definition.**
The **outer product** of $u$ (length $n$) and $v$ (length $m$) is the
$(n, m)$ matrix
$$u \otimes v \quad\text{with entries}\quad (u \otimes v)_{ij} = u_i\, v_j .$$
Contrast with the *inner* (dot) product, which eats two same-length
vectors and returns one number; the outer product eats any two vectors
and returns a whole grid.
In the banned-`@` register it is one broadcast:
`u[:, None] * v[None, :]` — shapes $(n,1)(1,m) \to (n,m)$.

**Worked example.**
$u = (2, -1)$, $v = (1, 3, 2)$:
$$u \otimes v = \begin{pmatrix} 2 & 6 & 4 \\ -1 & -3 & -2 \end{pmatrix}.$$
Look at the rows: row 0 is $2 \cdot v$, row 1 is $(-1) \cdot v$ — row
$i$ is always $u_i \cdot v$.
**Every row is a multiple of the same vector $v$** (and every column a
multiple of $u$): one independent direction.

**Rank of an atom.**
For nonzero $u, v$: $\operatorname{rank}(u \otimes v) = 1$ — the rows
span the line through $v$, no more (not 0: some $u_i \ne 0$ keeps row
$i$ nonzero).
Conversely, every rank-1 matrix *is* an outer product: all rows are
multiples of one direction $v$, and the multipliers stack into $u$.

**Recognizing and factoring.**
$$N = \begin{pmatrix} 3 & 6 \\ 1 & 2 \\ -2 & -4 \end{pmatrix}:$$
every row is a multiple of $(1, 2)$ — multipliers $3, 1, -2$ — so
$N = (3, 1, -2) \otimes (1, 2)$, certifying rank 1.
The factorization is not unique: $(6, 2, -4) \otimes (\tfrac12, 1)$
works too (scale trades freely between the factors).

In [6]:
def outer(u, v):
    return u[:, None] * v[None, :]


u = np.array([2., -1.])
v = np.array([1., 3., 2.])
P = outer(u, v)
print("u outer v:")
print(P)
print("row 0 / v:", P[0] / v, "   row 1 / v:", P[1] / v)   # constant: u_0, u_1

N = np.array([[3., 6.], [1., 2.], [-2., -4.]])
factor_gap = np.abs(N - outer(np.array([3., 1., -2.]), np.array([1., 2.]))).max()
print("factorization gap for N:", factor_gap)

u outer v:
[[ 2.  6.  4.]
 [-1. -3. -2.]]
row 0 / v: [2. 2. 2.]    row 1 / v: [-1. -1. -1.]
factorization gap for N: 0.0


### Checkpoint 5

1. By hand: compute $(1, -2) \otimes (3, 1, 0)$.
2. Factor $\begin{pmatrix} 2 & -2 \\ 4 & -4 \\ 6 & -6 \end{pmatrix}$ as
   an outer product $u \otimes v$, and state its rank.
3. What is the rank of $u \otimes v$ when $v = 0$ (and $u \ne 0$)?
   Why does the "rank 1" claim need both vectors nonzero?

## 6. Minimal Decompositions: How Few Atoms Suffice?

**Motivation.**
If rank-1 matrices are atoms, every matrix should be a molecule — a
*sum* of atoms.
How few atoms can build a given matrix?
The answer is beautiful: exactly the rank.
This is the deepest single idea in the unit, and the exam's hardest
matrix items live here.

**The rank bound.**
A sum of $k$ rank-1 pieces,
$M = u^{(1)} \otimes v^{(1)} + \cdots + u^{(k)} \otimes v^{(k)}$,
has $\operatorname{rank}(M) \le k$.
*Argument:* row $i$ of the sum is
$u^{(1)}_i v^{(1)} + \cdots + u^{(k)}_i v^{(k)}$ — a combination of the
same $k$ vectors $v^{(1)}, \dots, v^{(k)}$ for every row.
All rows live in a span of $k$ directions, so at most $k$ of them are
independent. $\square$

Flipped around: a rank-$r$ matrix can **never** be written with fewer
than $r$ atoms.
And $r$ atoms always suffice — the construction below shows how, and it
generalizes.

**Worked example: a minimal decomposition, start to finish.**
$$M = \begin{pmatrix} 1 & 1 & 0 \\ 2 & 0 & 1 \\ 4 & 2 & 1 \end{pmatrix}.$$

*Step 1 — find the rank.*
Rows: $r_0 = (1,1,0)$ and $r_1 = (2,0,1)$ are not multiples of each
other (look at the zeros) → at least 2.
Try building $r_2 = (4,2,1)$ from them: $a\,r_0 + b\,r_1 = (a + 2b,\; a,\; b)$;
matching gives $a = 2$, $b = 1$, and indeed
$2 r_0 + 1 r_1 = (4, 2, 1) = r_2$. ✓
So rank $= 2$.

*Step 2 — turn the row story into atoms.*
Every row is a combination of $r_0$ and $r_1$ with known coefficients:
$$\text{row } 0 = 1 \cdot r_0 + 0 \cdot r_1, \qquad
  \text{row } 1 = 0 \cdot r_0 + 1 \cdot r_1, \qquad
  \text{row } 2 = 2 \cdot r_0 + 1 \cdot r_1 .$$
Collect the $r_0$-coefficients into $a^{(1)} = (1, 0, 2)$ and the
$r_1$-coefficients into $a^{(2)} = (0, 1, 1)$:
$$M = a^{(1)} \otimes r_0 \;+\; a^{(2)} \otimes r_1 .$$
Two atoms.

*Step 3 — certify minimality.*
One atom would force rank $\le 1$, but $\operatorname{rank}(M) = 2$: the
rank bound says two is the floor.
Minimal. $\blacksquare$

The code verifies the decomposition to machine precision — broadcasting
only.

In [7]:
M = np.array([[1., 1., 0.], [2., 0., 1.], [4., 2., 1.]])

r0 = np.array([1., 1., 0.])
r1 = np.array([2., 0., 1.])
a1 = np.array([1., 0., 2.])     # each row's r0-coefficient
a2 = np.array([0., 1., 1.])     # each row's r1-coefficient

M_rebuilt = outer(a1, r0) + outer(a2, r1)
print("decomposition gap:", np.abs(M - M_rebuilt).max())    # ~0

# The rank-2 fact behind minimality, checked concretely:
print("r2 - (2 r0 + r1):", M[2] - (2 * r0 + r1))            # zeros

decomposition gap: 0.0
r2 - (2 r0 + r1): [0. 0. 0.]


### Checkpoint 6

1. A matrix is written as a sum of 3 outer products.
   What does that alone tell you about its rank — and can the rank be
   *less* than 3?
   (Hint: consider $u \otimes v + u \otimes w$.)
2. Write the identity matrix $I_2 = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix}$
   as a sum of outer products using as few atoms as possible, and
   justify that fewer is impossible.
3. A $5 \times 5$ matrix has rank 3.
   What is the minimum number of outer products in any decomposition,
   and which two facts combine to give that number?

## 7. Exam-Register Workout: Reconstruct, Sum, Decode

The real paper's multiple-choice items often wrap a Session-1 skill in a
**numeric normal form**: you compute an integer, normalize it as
$\pm m$ with $m \ge 0$, then decode via a stated rule so that exactly
one option is correct.
Here is a fully worked example in that register — this and the
banned-`@` Gram implementation worked in Session 2 §5 are the unit's
two dress rehearsals.

---

**Worked exam-style example 2 (multiple choice, numeric normal form).**

> A matrix $C \in \mathbb{R}^{3\times 2}$ acts on any $x = (x_0, x_1)$
> by
> $$C x = (3x_1,\;\; x_0 - 2x_1,\;\; 5x_0 + x_1).$$
> Reconstruct $C$, then compute $s$ = the sum of ALL entries of $C$.
> Your answer can be written as $s = \pm m$ with $m$ a non-negative
> integer.
> What is the value of $2m - 1$ if $s \ge 0$, or $2m$ if $s < 0$?
>
> A. 13  B. 14  C. 15  D. 16  E. 17
>
> Reasoning is not required.

*Solution, step by step.*

1. **Shape first.** Two inputs, three outputs → $C$ is $(3, 2)$.
2. **Read the rows.** Output 0 is $3x_1 = 0 \cdot x_0 + 3 \cdot x_1$
   → row $(0, 3)$ — the silent $x_0$ contributes a **zero entry**
   (Session 1, pitfall 2).
   Output 1: $(1, -2)$.
   Output 2: $(5, 1)$.
   $$C = \begin{pmatrix} 0 & 3 \\ 1 & -2 \\ 5 & 1 \end{pmatrix}.$$
3. **Sanity probe.** $C e_0$ should be $(0, 1, 5)$ — plug $x = (1, 0)$
   into the formula: $(0, 1, 5)$. ✓
4. **Sum the entries.** $s = 0 + 3 + 1 - 2 + 5 + 1 = 8$.
5. **Normal form.** $s = +8$, so $m = 8$ and $s \ge 0$.
6. **Decode.** $2m - 1 = 15$ → **C**.

The decode step is not decoration: with both branches on offer, a sign
slip in $s$ lands on a *different valid-looking option* ($2 \cdot 8 = 16$,
option D) instead of "not among the options" — the normal form is
engineered so that wrong work still decodes, wrongly.
Verify the recomputed value AND the decode, never just the letter.

In [8]:
# Verify the worked MC end to end.
C = np.array([[0., 3.], [1., -2.], [5., 1.]])


def c_formula(x):
    return np.array([3 * x[1], x[0] - 2 * x[1], 5 * x[0] + x[1]])


rng = np.random.default_rng(20260804)
xp = rng.normal(0, 1, 2)
print("reconstruction gap:", np.abs((C * xp).sum(axis=1) - c_formula(xp)).max())

s = C.sum()
m = abs(int(round(s)))
decoded = 2 * m - 1 if s >= 0 else 2 * m
print("s =", int(s), " m =", m, " decoded =", decoded, " -> option C")

reconstruction gap: 0.0
s = 8  m = 8  decoded = 15  -> option C


### Checkpoint 7

1. Same register, new action: $D \in \mathbb{R}^{3 \times 2}$ acts by
   $Dx = (x_0 - x_1,\;\; 2x_1,\;\; -3x_0)$.
   Reconstruct $D$, compute the entry sum $s$, and decode with the same
   rule ($2m - 1$ if $s \ge 0$, else $2m$).
2. In this decode rule, what value is reported when $s = 0$, and which
   branch fires?
   Why does a rule like this need the two branches to produce
   *disjoint* value sets?

## 8. Common Pitfalls III

**Pitfall 1 — counting nonzero entries instead of directions.**
"Lots of nonzero entries → high rank" and its converse are both wrong.
$\begin{pmatrix} 1 & 1 \\ 1 & 1 \end{pmatrix}$ has four nonzeros and
rank **1** (rows identical); $\begin{pmatrix} 0 & 5 \\ 7 & 0 \end{pmatrix}$
has two nonzeros and rank **2** (rows not multiples).
Rank counts *independent directions*, not ink:

In [9]:
dense = np.array([[1., 1.], [1., 1.]])
sparse = np.array([[0., 5.], [7., 0.]])

print("dense rows ratio  :", dense[1] / dense[0], " -> constant: rank 1")
print("sparse row multiple? (0,5)*c can never hit (7,0) -> rank 2")
# and the collision test agrees: dense crushes (1,-1), sparse crushes nothing
print("dense @ (1,-1)    :", (dense * np.array([1., -1.])).sum(axis=1))

dense rows ratio  : [1. 1.]  -> constant: rank 1
sparse row multiple? (0,5)*c can never hit (7,0) -> rank 2
dense @ (1,-1)    : [0. 0.]


Fix: always argue rank via builds ("row 2 = row 0 + row 1") or their
impossibility — never via visual density.

**Pitfall 2 — flipping "orthogonal ⇒ independent".**
The true direction: nonzero, pairwise-orthogonal vectors *are*
independent.
(Sketch: dot any zero-combination with one of the vectors — F2
orthogonality kills every term but its own, forcing that coefficient to
zero.)
The **converse is false**: independent vectors need not be orthogonal —
$(1, 0)$ and $(1, 1)$ are independent (not multiples) yet their dot
product is $1 \ne 0$.
"Not orthogonal" never certifies dependence:

In [10]:
p, q = np.array([1., 0.]), np.array([1., 1.])
print("dot:", p @ q, " -> not orthogonal")
# Independence anyway: c*p = q would need (c, 0) = (1, 1) -- impossible.
print("q - 1.0 * p:", q - p, " -> nonzero: q is not a multiple of p")

dot: 1.0  -> not orthogonal
q - 1.0 * p: [0. 1.]  -> nonzero: q is not a multiple of p


Fix: keep the implication one-way in your notes — orthogonal (and
nonzero) ⇒ independent; independent ⇏ orthogonal.

**Pitfall 3 — reaching for an inverse computation.**
Asked "is $Z$ invertible?", a student starts hunting for an inverse
formula or a library call.
Out of scope here, banned in the exam's register (`np.linalg` is on the
zero-points lists), and *unnecessary*: invertibility questions in this
course are rank questions.
Broken instinct: "compute the undo, see if it exists."
Fixed instinct: "count independent directions; full rank ⇔ undo
exists" — argued with builds and collisions like Section 4, no inverse
entries ever produced:

In [11]:
Z = np.array([[1., 2., 0.], [0., 1., 1.], [1., 4., 2.]])
# Rank reasoning, no inverse computation: is row2 buildable from rows 0, 1?
# a*(1,2,0) + b*(0,1,1) = (a, 2a+b, b); match (1,4,2): a=1, b=2, 2a+b=4. Yes.
print("row2 - (1*row0 + 2*row1):", Z[2] - (Z[0] + 2 * Z[1]))   # zeros
print("-> rank 2 < 3: NOT invertible (no inverse was ever computed)")

row2 - (1*row0 + 2*row1): [0. 0. 0.]
-> rank 2 < 3: NOT invertible (no inverse was ever computed)


### Checkpoint 8

1. A $3 \times 3$ matrix has exactly three nonzero entries.
   Give one such matrix with rank 3 and one with rank 1.
2. True or false, with one line each:
   (a) pairwise-orthogonal nonzero vectors are independent;
   (b) independent vectors are pairwise orthogonal;
   (c) to decide whether a $3 \times 3$ matrix is invertible you must
   compute its inverse.

## Exam Connections

How this unit's material shows up in Round 1 (paraphrased from the
`reference/analysis.md` topic table — no real test text here):

- The **linear-algebra cluster is the largest on the paper**: 10
  sub-parts worth 60 of the 300 points in r1-2026 (vectors, projection,
  rank, outer products, Gram matrix, and decomposition topics).
  The analysis's difficulty profile marks a further ~60 points as
  reachable once a linear-algebra foundation (dot products, norms,
  matrix action, rank) is in place — this unit is that foundation.
- **Matrix-from-action is the signature pattern**: an action is
  described (formula or probe results), and later parts *consume* the
  reconstruction — a rank read feeds an outer-product decomposition
  feeds a verification, exactly the texture of p13 and p17's arcs.
  Sessions 1 §5–6 and this session's §5–7 are the drill line.
- The **NumPy implementation cluster** (8 sub-parts, 55 points) carries
  the banned-API register: explicit zero-points clauses naming `@`,
  transpose-style calls, `np.dot`, and loops, with
  broadcasting-and-axis-sums as the intended route — Session 2 §4–5's
  exact shape, drilled by p05, p07, p09, p10, p14, p16.
- Multiple-choice items use exactly five options A–E, and numeric
  answers arrive in the **normal form** register (sign convention plus
  decode rule) worked in §7 — p04 trains it.

If your budget is tight, the highest-yield hour in this unit is:
matrix-from-action reconstruction (both directions), one banned-`@`
Gram implementation, and rank-by-inspection reads.

## Going Deeper

Optional forward pointers along the course map — nothing here is needed
for this unit's practice:

- **`F6-svd-spectral`**: this session decomposed matrices into rank-1
  atoms *exactly*; F6 asks the sharper question — if you may keep only
  $k$ atoms, which choice approximates a matrix *best*?
  The answer machinery (and the exam's advanced-tier items) build
  directly on rank and outer products as taught here.
- **`C8-embeddings`**: stacks of row vectors with cosine-normalized
  Gram tables — Session 2 §5–6 at industrial scale, where "which rows
  point the same way" becomes the organizing question of an entire
  application area.

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. Dependent: $(6, 2) = 2 \cdot (3, 1)$.
2. $(1, 1, 1) = (1, 0, 1) + (0, 1, 0)$ — the third is built from the
   first two.
3. The zero vector is always buildable from the others with all-zero
   coefficients ($0 = 0 \cdot v_1 + \cdots$), so "some listed vector is
   a combination of the rest" holds automatically.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. The $xz$-plane: all vectors $(a, 0, b)$.
   $(2, 0, -5)$ ✓ (middle entry 0); $(0, 1, 0)$ ✗ (no combination can
   produce the middle 1).
2. Test: $y_2 = 3y_1 - 5y_0 \to 3 \cdot 1 - 5 \cdot 2 = -7 = y_2$ ✓
   reachable.
3. Nothing — $u + 2w$ was already reachable, so the reachable set (the
   plane) is unchanged; only the list got longer.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. Rank 1: row 1 $= 2 \cdot$ row 0.
2. Rank 2: row 2 $=$ row 0 $+$ row 1; rows 0 and 1 are not multiples of
   each other.
3. Rank $\le \min(3, 5) = 3$ — the *row* side: only three rows exist,
   so at most three independent directions.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. Columns $(1, 2)$ and $(3, 6)$: the second is $3 \times$ the first →
   rank 1 → not invertible.
   Collision: $x = (3, -1)$ gives $3c_0 - c_1 = 0$.
2. If $Au = Av$ with $u \ne v$, then $A(u - v) = 0$ writes a *nonzero*
   zero-combination of the columns — impossible when they are
   independent (full rank).
3. Never: $\operatorname{rank}(AB) \le \min(\operatorname{rank} A,
   \operatorname{rank} B) \le 3 < 5$ — a rank-deficient $5 \times 5$
   matrix, not invertible, for every choice of $A, B$.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. $\begin{pmatrix} 3 & 1 & 0 \\ -6 & -2 & 0 \end{pmatrix}$.
2. Every row is a multiple of $(1, -1)$ with multipliers $(2, 4, 6)$:
   $u = (2, 4, 6)$, $v = (1, -1)$ (any rescaling works).
   Rank 1 — it *is* a single outer product of nonzero vectors.
3. Rank 0 — the outer product is the zero matrix.
   With $v = 0$ (or $u = 0$) every entry $u_i v_j$ dies, so "rank
   exactly 1" genuinely needs both factors nonzero.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. Rank $\le 3$ (the rank bound).
   Yes, it can be less: $u \otimes v + u \otimes w = u \otimes (v + w)$
   has rank $\le 1$ — atoms can overlap or cancel.
2. $I_2 = e_0 \otimes e_0 + e_1 \otimes e_1$ — two atoms.
   Fewer is impossible: one atom means rank $\le 1$, but
   $\operatorname{rank}(I_2) = 2$ (independent rows).
3. Three — the rank bound forbids fewer than $\operatorname{rank} = 3$,
   and the Step-2 construction (rows as combinations of 3 independent
   rows) always achieves exactly $\operatorname{rank}$ many.

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. $D = \begin{pmatrix} 1 & -1 \\ 0 & 2 \\ -3 & 0 \end{pmatrix}$
   (two zero entries!); $s = 1 - 1 + 0 + 2 - 3 + 0 = -1$, so $m = 1$
   and $s < 0$: decode $2m = 2$.
2. $s = 0$ gives $m = 0$ and fires the $s \ge 0$ branch:
   $2 \cdot 0 - 1 = -1$.
   Disjoint branch outputs (odd for $s \ge 0$, even for $s < 0$) make
   the letter decodable *uniquely* from the reported value — if both
   branches could emit the same number, two different computations
   would be indistinguishable and the item ungradable.

</details>

<details><summary><b>Checkpoint 8</b></summary>

1. Rank 3: $\mathrm{diag}(1, 1, 1)$ — three independent rows.
   Rank 1: all three nonzeros in one row, e.g. row 0 $= (1, 2, 3)$,
   rows 1–2 zero.
2. (a) True — dot a zero-combination with each vector; orthogonality
   isolates its coefficient, forcing it to 0.
   (b) False — $(1, 0)$ and $(1, 1)$: independent, dot product 1.
   (c) False — full rank ⇔ invertible; rank is decided by
   independence reasoning, no inverse entries needed (and none are
   taught here).

</details>